``AAPred().eval_selective()`` measures the **risk-coverage trade-off**: how much a model's performance improves when it only has to score its most-confident samples. Samples are ranked by a per-sample confidence signal, and every metric is scored again on the most-confident ``coverage`` fraction of them (see [Breimann25]_). It is a *measurement*: nothing abstains, and choosing a refusal threshold from the curve stays with the caller. We first obtain the ``DOM_GSEC`` dataset and its feature matrix:

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False  # Disable verbosity

# DOM_GSEC example dataset + its feature set (see [Breimann25]_)
df_seq = aa.load_dataset(name="DOM_GSEC")
labels = df_seq["label"].to_list()
df_feat = aa.load_features(name="DOM_GSEC").head(20)

# Build the CPP feature matrix
sf = aa.SequenceFeature()
df_parts = sf.get_df_parts(df_seq=df_seq)
X = sf.feature_matrix(features=df_feat["feature"], df_parts=df_parts)

``eval_selective`` cross-validates the models given at construction (no prior :meth:`fit` is needed) and returns one row per metric and coverage level. ``score`` is the metric on the retained subset, ``n_retained`` the number of samples it kept, and ``score_aurc`` the area under that metric's coverage-performance curve. The ``coverage = 1.0`` row retains every sample, so it is the ordinary out-of-fold score:

In [2]:
aap = aa.AAPred(models=["rf", "extra_trees", "log_reg"], random_state=42)
df_eval_selective = aap.eval_selective(X, labels)
aa.display_df(df_eval_selective, n_rows=10, show_shape=True)

DataFrame shape: (20, 5)


,metric,coverage,n_retained,score,score_aurc
1,accuracy,0.206349,26,0.961538,0.897369
2,accuracy,0.404762,51,0.941176,0.897369
3,accuracy,0.603175,76,0.881579,0.897369
4,accuracy,0.801587,101,0.881188,0.897369
5,accuracy,1.000000,126,0.809524,0.897369
6,balanced_accuracy,0.206349,26,0.954545,0.898337
7,balanced_accuracy,0.404762,51,0.942742,0.898337
8,balanced_accuracy,0.603175,76,0.884778,0.898337
9,balanced_accuracy,0.801587,101,0.883794,0.898337
10,balanced_accuracy,1.000000,126,0.809524,0.898337


``metrics`` selects which performance metrics are scored, and ``coverages`` sets the grid of retained fractions (each in ``(0, 1]``, strictly increasing). A finer grid traces the curve more closely:

In [3]:
df_eval_selective = aap.eval_selective(X, labels,
                                       metrics=["balanced_accuracy", "f1"],
                                       coverages=[0.25, 0.5, 0.75, 1.0])
aa.display_df(df_eval_selective, n_rows=10, show_shape=True)

DataFrame shape: (8, 5)


,metric,coverage,n_retained,score,score_aurc
1,balanced_accuracy,0.253968,32,0.933333,0.891652
2,balanced_accuracy,0.500000,63,0.907484,0.891652
3,balanced_accuracy,0.753968,95,0.895722,0.891652
4,balanced_accuracy,1.000000,126,0.809524,0.891652
5,f1,0.253968,32,0.916667,0.880873
6,f1,0.500000,63,0.888889,0.880873
7,f1,0.753968,95,0.888889,0.880873
8,f1,1.000000,126,0.812500,0.880873


``confidence`` is the ranking signal and is fully pluggable: pass the score margin, an uncertainty measure, or an applicability-domain distance (negated, so that larger means more confident). Here the ensemble's disagreement (``score_std`` from :meth:`AAPred.predict_oof`) is negated, so samples the models agree on are retained first. Leaving ``confidence=None`` ranks by the out-of-fold score margin instead:

In [4]:
df_pred = aap.predict_oof(X, labels)
confidence = -df_pred["score_std"].to_numpy()  # low disagreement = high confidence
df_eval_selective = aap.eval_selective(X, labels,
                                       confidence=confidence,
                                       metrics=["accuracy"],
                                       coverages=[0.25, 0.5, 0.75, 1.0])
aa.display_df(df_eval_selective, n_rows=10, show_shape=True)

DataFrame shape: (4, 5)


,metric,coverage,n_retained,score,score_aurc
1,accuracy,0.253968,32,0.781250,0.821099
2,accuracy,0.500000,63,0.825397,0.821099
3,accuracy,0.753968,95,0.842105,0.821099
4,accuracy,1.000000,126,0.809524,0.821099


``n_cv`` sets the number of stratified cross-validation folds behind the out-of-fold scores (it must not exceed the smallest class count), and ``label_pos`` names the class scored as positive:

In [5]:
df_eval_selective = aap.eval_selective(X, labels,
                                       metrics=["accuracy"],
                                       n_cv=3,
                                       label_pos=1)
aa.display_df(df_eval_selective, n_rows=10, show_shape=True)

DataFrame shape: (5, 5)


,metric,coverage,n_retained,score,score_aurc
1,accuracy,0.206349,26,0.923077,0.884192
2,accuracy,0.404762,51,0.921569,0.884192
3,accuracy,0.603175,76,0.881579,0.884192
4,accuracy,0.801587,101,0.871287,0.884192
5,accuracy,1.000000,126,0.801587,0.884192
